# Sidekick Policy Training (vocab1 / 1793, ~4.1M params)

Trains a **small policy transformer from scratch** with the same architecture code as production
(`policy_transformer/model.py`: RoPE causal attention, GELU MLP, untied `lm_head`), scaled down to
`n_embd=256, n_head=8, n_layer=4` (≈4.1M params vs ≈25.9M for the production config on vocab1).
Same data and batch mix as the vocab1 run. Purpose: a sidekick policy to guide MCTS below the root,
with the large policy used only at the root and the value network unchanged.

Vocab1 (1793) = every queen/knight-geometry from→to pair (1792) + pad, a strict subset of production `vocab.pkl` (4033),
so every sidekick move maps to a production move by UCI string.

## Upload

Reuse the Kaggle Dataset from the vocab1 run if you have it. Required files:

| Local path | Upload as |
|---|---|
| `data/full_X.pt`, `data/full_Y.pt` | `full_X.pt`, `full_Y.pt` |
| `data/weak_X.pt`, `data/weak_Y.pt` | `weak_X.pt`, `weak_Y.pt` |
| `data/elite_X.pt`, `data/elite_Y.pt` | `elite_X.pt`, `elite_Y.pt` |
| `policy_transformer/vocab1.pkl` | `vocab1.pkl` |

`ultra_3o9.pt` and `vocab.pkl` are **not** needed (no warm-start).

## Run overnight

1. Settings: Accelerator **GPU T4** (one GPU is used), Persistence off is fine.
2. Add Data → attach the dataset.
3. Click **Save Version → Save & Run All (Commit)**. This runs in the background, so you can close the browser.
   An interactive session stops when idle; a committed run does not.
4. Training runs `run_iters` steps (default 100k, cosine LR over 100k), saving `sidekick_v1.pt` every 1k steps.
   Check the log after ~10 minutes: if `it/s` × 3600 × 11 < `run_iters`, cancel and lower `run_iters` and `lr_decay_iters` together
   so the run finishes inside Kaggle's 12 h limit.
5. In the morning: open the version → **Output** → download `best_sidekick_v1.pt` (best val loss) and `sidekick_v1.pt` (last).
   The log ends with val cross-entropy and top-1 / top-7 next-move accuracy (against the move actually played, not Stockfish).


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torch.nn import functional as F
import math


def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary position embedding to a tensor shaped [B, n_head, T, head_dim].
    cos/sin are shaped [T, head_dim//2] (broadcasted inside).
    """
    # x: [..., head_dim] where head_dim is even
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos
    return torch.stack((out_even, out_odd), dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        head_dim = n_embd // n_head
        assert head_dim % 2 == 0, "RoPE requires an even head dimension"

        # RoPE (rotary position embeddings) cache (non-persistent so old checkpoints still load)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("_rope_inv_freq", inv_freq, persistent=False)
        self.register_buffer("_rope_cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("_rope_sin_cached", torch.empty(0), persistent=False)
        self._rope_seq_len_cached = 0
        self._rope_cache_dtype = None
        self._rope_cache_device = None

        # Flash Attention requires PyTorch 2.0+
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: Flash Attention not available. Using slow manual attention.")
            self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                        .view(1, 1, block_size, block_size))

    def _rope_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        # Cache per (device, dtype, seq_len>=T). Compute in fp32 for stability, cast for use.
        if (
            self._rope_seq_len_cached >= seq_len
            and self._rope_cache_device == device
            and self._rope_cache_dtype == dtype
            and self._rope_cos_cached.numel() != 0
        ):
            return self._rope_cos_cached[:seq_len], self._rope_sin_cached[:seq_len]

        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self._rope_inv_freq.to(device=device))  # [T, head_dim//2]
        cos = freqs.cos().to(dtype=dtype)
        sin = freqs.sin().to(dtype=dtype)

        self._rope_cos_cached = cos
        self._rope_sin_cached = sin
        self._rope_seq_len_cached = seq_len
        self._rope_cache_device = device
        self._rope_cache_dtype = dtype
        return cos, sin

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # RoPE: apply rotary embeddings to q/k (not v)
        cos, sin = self._rope_cos_sin(T, device=q.device, dtype=q.dtype)
        q = _apply_rope(q, cos, sin)
        k = _apply_rope(k, cos, sin)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
            )
        else:
            # manual implementation (fallback)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedFoward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class PolicyHead(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout, device):
        super().__init__()
        self.block_size = block_size
        self.device = device
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Ensure T is within block_size
        if T > self.block_size:
            idx = idx[:, -self.block_size:]
            T = self.block_size
            if targets is not None:
                targets = targets[:, -self.block_size:]

        tok_emb = self.token_embedding_table(idx) 
        # With RoPE applied inside attention, we don't add absolute positional embeddings here.
        x = tok_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x[:, [-1], :]) # optimization: only compute last logit for inference
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-100) # Added ignore_index explicitly
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
# ───────────────────────────────────────────────────────────────
# KAGGLE DATA SETUP: auto-find + symlink training files
# ───────────────────────────────────────────────────────────────
import os

input_root = "/kaggle/input"

REQUIRED_FILES = {
    "vocab1.pkl": "vocab1.pkl",
    "full_X.pt": "full_X.pt",
    "full_Y.pt": "full_Y.pt",
    "elite_X.pt": "elite_X.pt",
    "elite_Y.pt": "elite_Y.pt",
    "weak_X.pt": "weak_X.pt",
    "weak_Y.pt": "weak_Y.pt",
}

# Only to recover from a crashed run: set to an uploaded sidekick checkpoint filename.
RESUME_CHECKPOINT = None  # e.g. "sidekick_v1.pt"

OPTIONAL_FILES = {RESUME_CHECKPOINT: RESUME_CHECKPOINT} if RESUME_CHECKPOINT else {}
files_to_link = {**REQUIRED_FILES, **OPTIONAL_FILES}
found = {local: False for local in files_to_link.values()}

for root, dirs, files in os.walk(input_root):
    for original_name, local_name in files_to_link.items():
        if original_name in files and not found[local_name]:
            src = os.path.join(root, original_name)
            if os.path.lexists(local_name):
                os.remove(local_name)
            os.symlink(src, local_name)
            found[local_name] = True
            print(f"Linked {src} → ./{local_name}")

missing = [f for f in REQUIRED_FILES.values() if not found[f]]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing) + ". Attach the dataset via 'Add Data'.")
print("All required files linked.")


## Train

Iteration-based, same scheme as the vocab1 notebook: linear warmup for `warmup_iters`, then cosine decay to `min_lr`
at `lr_decay_iters`. Checkpoints: `sidekick_v1.pt` every `save_interval` steps, `best_sidekick_v1.pt` on each val improvement.
Re-run this cell to continue from a checkpoint; set `RESUME_CHECKPOINT` in the link cell first.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Train sidekick policy (vocab1, from scratch)
# ──────────────────────────────────────────────────────────────
import math, os, pickle, gc, time

import torch
from torch.amp import GradScaler, autocast
from tqdm import tqdm

from model import PolicyHead

# ── Architecture: production code, scaled down ──
n_embd = 256
n_head = 8
n_layer = 4
block_size = 128
dropout = 0.1

# ── Optimisation (original from-scratch recipe: peak 3e-4, min 3e-5, warmup 1k) ──
learning_rate = 3e-4
min_lr = learning_rate / 10
warmup_iters = 1_000
weight_decay = 1e-4
grad_accum = 1

# ── Run control (iteration-based, as in the vocab1 notebook) ──
max_iters = 100_001
run_iters = 100_000         # how many steps THIS session will run
lr_decay_iters = 100_000
eval_interval = 1_000
eval_iters = 50
save_interval = 1_000
log_interval = 500

# Batch mixture (same as the vocab1 run)
n_normal = 256
n_elite = 32
n_weak = 32
batch_size = n_normal + n_elite + n_weak  # 320

output_checkpoint = "sidekick_v1.pt"
checkpoint_path = RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT)) else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable a GPU accelerator in the Kaggle notebook settings."
scaler = GradScaler(enabled=True)

# ── Vocab ──
with open("vocab1.pkl", "rb") as f:
    stoi = pickle.load(f)
vocab_size = len(stoi)
assert "|" in stoi, "vocab1.pkl missing pad token '|'"
assert vocab_size == 1793, f"Expected vocab1 size 1793, got {vocab_size}."
print(f"Vocab size: {vocab_size}")

# ── Data ──
def load_and_split(x_file, y_file, train_ratio=0.9):
    print(f"Loading {x_file} / {y_file} ...")
    X = torch.load(x_file, map_location="cpu", weights_only=False)
    Y = torch.load(y_file, map_location="cpu", weights_only=False)
    assert X.shape == Y.shape, f"Shape mismatch: {X.shape} vs {Y.shape}"
    assert int(X.min()) >= 0 and int(X.max()) < vocab_size, "X not tokenized with vocab1.pkl"
    y_pos = Y[Y >= 0]
    assert len(y_pos) == 0 or int(y_pos.max()) < vocab_size, "Y not tokenized with vocab1.pkl"
    split = int(train_ratio * len(X))
    out = (X[:split], Y[:split], X[split:], Y[split:])
    del X, Y
    gc.collect()
    print(f"  train tokens {len(out[0]):,}, val tokens {len(out[2]):,}")
    return out

X_train, Y_train, X_val, Y_val = load_and_split("full_X.pt", "full_Y.pt")
X_train_weak, Y_train_weak, X_val_weak, Y_val_weak = load_and_split("weak_X.pt", "weak_Y.pt")
X_train_elite, Y_train_elite, X_val_elite, Y_val_elite = load_and_split("elite_X.pt", "elite_Y.pt")


def get_batch(split):
    if split == "train":
        sets = [(X_train, Y_train, n_normal), (X_train_elite, Y_train_elite, n_elite), (X_train_weak, Y_train_weak, n_weak)]
    else:
        sets = [(X_val, Y_val, n_normal), (X_val_elite, Y_val_elite, n_elite), (X_val_weak, Y_val_weak, n_weak)]
    xs, ys = [], []
    for dx, dy, n in sets:
        upper = max(1, len(dx) - block_size + 1)
        ix = torch.randint(0, upper, (n,))
        xs.append(torch.stack([dx[i : i + block_size] for i in ix]))
        ys.append(torch.stack([dy[i : i + block_size] for i in ix]))
    x = torch.cat(xs).to(device, dtype=torch.long, non_blocking=True)
    y = torch.cat(ys).to(device, dtype=torch.long, non_blocking=True)
    return x, y


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            with autocast(device_type="cuda"):
                _, loss = model(xb, yb)
            losses[k] = loss
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def topk_accuracy(model, n_batches=200, ks=(1, 7)):
    # Next-move accuracy on val against the move actually played (not Stockfish's best move).
    model.eval()
    hits = {k: 0 for k in ks}
    total = 0
    for _ in range(n_batches):
        xb, yb = get_batch("val")
        with autocast(device_type="cuda"):
            logits, _ = model(xb, yb)          # (B*T, V)
        targets = yb.view(-1)
        mask = targets != -100
        logits, targets = logits[mask].float(), targets[mask]
        top = logits.topk(max(ks), dim=-1).indices
        for k in ks:
            hits[k] += (top[:, :k] == targets[:, None]).any(dim=-1).sum().item()
        total += targets.numel()
    model.train()
    return {k: hits[k] / max(total, 1) for k in ks}


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) * (learning_rate - min_lr)


def save_checkpoint(model, optimizer, iter_num, best_val_loss, path):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "iter_num": iter_num,
        "best_val_loss": best_val_loss,
        "config": {"vocab_size": vocab_size, "n_embd": n_embd, "n_head": n_head,
                   "n_layer": n_layer, "block_size": block_size, "dropout": dropout},
    }, path)
    print(f"Saved {path} (iter {iter_num})")


def main():
    torch.set_float32_matmul_precision("high")
    model = PolicyHead(vocab_size, n_embd, block_size, n_head, n_layer, dropout, device).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    iter_num, best_val_loss = 0, 1e9
    if checkpoint_path:
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        iter_num = int(ckpt["iter_num"])
        best_val_loss = float(ckpt.get("best_val_loss", 1e9))
        print(f"Resumed at iter {iter_num}, best_val_loss={best_val_loss:.4f}")

    print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
    print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    print(f"Batch: {batch_size} (normal={n_normal}, elite={n_elite}, weak={n_weak}), block {block_size}")
    target_end_iter = min(max_iters, iter_num + run_iters)
    print(f"Will run iters [{iter_num} → {target_end_iter})")
    start_iter = iter_num

    t0 = time.time()
    model.train()
    pbar = tqdm(total=target_end_iter - iter_num, desc="Training", unit="iter", dynamic_ncols=True, mininterval=30)

    while iter_num < target_end_iter:
        elapsed = time.time() - t0
        lr = get_lr(iter_num)
        for g in optimizer.param_groups:
            g["lr"] = lr

        if iter_num > 0 and iter_num % eval_interval == 0:
            losses = estimate_loss(model)
            print(f"step {iter_num}: train {losses['train']:.4f}, val {losses['val']:.4f}, "
                  f"lr {lr:.2e}, {elapsed/3600:.2f} h, {(iter_num-start_iter)/max(elapsed,1):.1f} it/s")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, optimizer, iter_num, best_val_loss, f"best_{output_checkpoint}")

        optimizer.zero_grad(set_to_none=True)
        accum = 0.0
        for _ in range(grad_accum):
            xb, yb = get_batch("train")
            with autocast(device_type="cuda"):
                _, loss = model(xb, yb)
                loss = loss / grad_accum
            scaler.scale(loss).backward()
            accum += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if iter_num % log_interval == 0:
            print(f"step {iter_num}: loss {accum:.4f}, lr {lr:.2e}")
        iter_num += 1
        pbar.update(1)
        if iter_num % save_interval == 0:
            save_checkpoint(model, optimizer, iter_num, best_val_loss, output_checkpoint)

    pbar.close()
    save_checkpoint(model, optimizer, iter_num, best_val_loss, output_checkpoint)
    losses = estimate_loss(model)
    if losses["val"] < best_val_loss:
        best_val_loss = losses["val"]
        save_checkpoint(model, optimizer, iter_num, best_val_loss, f"best_{output_checkpoint}")
    acc = topk_accuracy(model)
    print("=" * 60)
    print(f"Finished at iter {iter_num} ({(time.time()-t0)/3600:.2f} h)")
    print(f"Final: train CE {losses['train']:.4f}, val CE {losses['val']:.4f}, best val CE {best_val_loss:.4f}")
    print(f"Val next-move accuracy (vs played move): top-1 {acc[1]:.3f}, top-7 {acc[7]:.3f}")
    print(f"Download best_{output_checkpoint} and {output_checkpoint} from the version Output tab.")


main()
